# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects")

print("Path to dataset files:", path)

100%|██████████| 18.7M/18.7M [01:08<00:00, 284kB/s]

Extracting files...


Path to dataset files: C:\Users\ladol\.cache\kagglehub\datasets\gzdekzlkaya\wikipedia-text-corpus-for-nlp-and-llm-projects\versions\1


Cargar el corpus

In [46]:
import pandas as pd
import os

ruta_wikipedia = os.path.join(os.getcwd(), 'wikipedia_text_corpus.csv')

# 2. Cargar el corpus usando Pandas
# index_col=0 le indica que la primera columna (los números) es el índice
df_wiki = pd.read_csv(ruta_wikipedia, index_col=0)

# 3. Renombrar la columna a 'raw' para seguir tu estándar de clase
df_wiki = df_wiki.rename(columns={'text': 'raw'})

# 4. Limpieza preventiva: Eliminar filas vacías que podrían dar error en el preprocesamiento
df_wiki = df_wiki.dropna(subset=['raw'])

print(f'Número de documentos (artículos completos): {len(df_wiki)}')
print(f'\nEjemplo del primer documento (primeros 200 caracteres):')
print(df_wiki['raw'].iloc[0][:200])

Número de documentos (artículos completos): 10859

Ejemplo del primer documento (primeros 200 caracteres):
Anovo

Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It was founded in 1987, went public in 1999, and is currently a member of the CAC Small.

It won in the categor


Stopwords

In [47]:
import nltk
from nltk.corpus import stopwords

# nltk.download('stopwords')

STOPWORDS_NLTK = set(stopwords.words('english'))

print(f'Stopwords en NLTK: {len(STOPWORDS_NLTK)} palabras')

Stopwords en NLTK: 198 palabras


In [48]:
import math
import re
import string
from collections import Counter, defaultdict
def limpiar_texto(texto):
    texto = texto.lower()
    # Eliminar puntuación y caracteres especiales (dejar solo letras y espacios)
    texto = re.sub(r'[^a-z\s]', '', texto)
    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


In [49]:
def tokenizar(texto, stopwords=STOPWORDS_NLTK):
    texto_limpio = limpiar_texto(texto)
    tokens = texto_limpio.split()
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]

    return tokens

In [50]:
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer('english')

In [51]:
import pandas as pd

In [52]:
def procesar(texto):
    # Validar que el texto sea una cadena; si es nulo (NaN), retornar string vacío
    if not isinstance(texto, str):
        return ""
    
    tokens = tokenizar(texto) 
    stems = [stemmer.stem(t) for t in tokens]
    return " ".join(stems)

Con el dataframe definido vamos a procesar

In [53]:
df_wiki['processed'] = df_wiki['raw'].apply(procesar)

print(df_wiki[['raw', 'processed']].head())

                                                 raw  \
1  Anovo\n\nAnovo (formerly A Novo) is a computer...   
2  Battery indicator\n\nA battery indicator (also...   
3  Bob Pease\n\nRobert Allen Pease (August 22, 19...   
4  CAVNET\n\nCAVNET was a secure military forum w...   
5  CLidar\n\nThe CLidar is a scientific instrumen...   

                                           processed  
1  anovo anovo former novo comput servic compani ...  
2  batteri indic batteri indic also known batteri...  
3  bob peas robert allen peas august june analog ...  
4  cavnet cavnet secur militari forum becam oper ...  
5  clidar clidar scientif instrument use measur p...  


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

TFIDF

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Instanciar el vectorizador
vectorizador = TfidfVectorizer(lowercase=False)

# Esto convertirá la columna 'processed' en una matriz de (10859, número_de_palabras)
tfidf_matrix = vectorizador.fit_transform(df_wiki['processed'])

print(f"Dimensiones de la matriz (Documentos, Vocabulario): {tfidf_matrix.shape}")

Dimensiones de la matriz (Documentos, Vocabulario): (10859, 135397)


In [62]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recuperar_tfidf(query, top_n=10):
    query_p = procesar(query)
    query_vector = vectorizador.transform([query_p])
    similitudes = cosine_similarity(tfidf_matrix, query_vector).flatten()
    
    # 4. Encontrar los índices de los mejores resultados
    indices_mejores = similitudes.argsort()[-top_n:][::-1]
    
    # 5. Retornar los documentos correspondientes del DataFrame
    resultados = df_wiki.iloc[indices_mejores].copy()
    resultados['score_similitud'] = similitudes[indices_mejores]
    
    return resultados[['raw', 'score_similitud']]

In [56]:
queries_taller = [
    "Renewable methanol production in Iceland",
    "Cryptocurrency legal tender in Marshall Islands",
    "Self-propelled mower Claas Cougar",              
    "Conductive organic polymers and OLEDs",
    "Chemical Agent Resistant Coating military",
    "Electronic money system in Ecuador",       
    "Analog integrated circuit design Bob Pease",     
    "Ammeter and voltmeter battery indication",       
    "Capacity fading in lithium-ion batteries",       
    "Secure military forum Iraq combat"               
]

In [63]:

print(f"{'Query del Usuario':<40} | {'Documento Más Relevante':<25} | {'Score'}")

for q in queries_taller:
    res = recuperar_tfidf(q, top_n=1)
    titulo_doc = res['raw'].values[0].split('\n')[0][:25]
    score = res['score_similitud'].values[0]
    print(f"{q:<40} | {titulo_doc:<25} | {score:.4f}")

Query del Usuario                        | Documento Más Relevante   | Score
Renewable methanol production in Iceland | Carbon Recycling Internat | 0.5799
Cryptocurrency legal tender in Marshall Islands | Double-spending           | 0.2802
Self-propelled mower Claas Cougar        | Claas Cougar              | 0.8171
Conductive organic polymers and OLEDs    | OLED                      | 0.6458
Chemical Agent Resistant Coating military | Conversion coating        | 0.3865
Electronic money system in Ecuador       | Money order               | 0.3389
Analog integrated circuit design Bob Pease | Bob Pease                 | 0.6544
Ammeter and voltmeter battery indication | Battery indicator         | 0.5216
Capacity fading in lithium-ion batteries | Capacity loss             | 0.5341
Secure military forum Iraq combat        | Broadband Forum           | 0.3268


## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [64]:
from rank_bm25 import BM25Okapi

Adaptar el contenido de tu DataFrame al formato BM25
Reutilizamos la columna 'processed' que ya tiene el Stemming hecho

In [65]:
corpus_para_bm25 = [doc.split() for doc in df_wiki['processed']]

In [66]:
bm25 = BM25Okapi(corpus_para_bm25)

Función de recuperación BM25 (Actividad 6)

In [70]:
def recuperar_bm25(query, top_n=10):
    query_tokens = procesar(query).split()
    puntajes = bm25.get_scores(query_tokens)
    
    indices_mejores = puntajes.argsort()[-top_n:][::-1]
    
    resultados = df_wiki.iloc[indices_mejores].copy()
    resultados['score_bm25'] = puntajes[indices_mejores]
    
    return resultados[['raw', 'score_bm25']]

In [72]:
for q in queries_taller:
    res = recuperar_bm25(q, top_n=1)
    titulo = res['raw'].values[0].split('\n')[0][:25]
    score = res['score_bm25'].values[0]
    print(f"{q:<40} | {titulo:<25} | {score:.4f}")

Renewable methanol production in Iceland | Carbon Recycling Internat | 29.8706
Cryptocurrency legal tender in Marshall Islands | Digital currency          | 24.7013
Self-propelled mower Claas Cougar        | Claas Cougar              | 59.2773
Conductive organic polymers and OLEDs    | Organic electronics       | 26.6459
Chemical Agent Resistant Coating military | Chemical Agent Resistant  | 28.2861
Electronic money system in Ecuador       | Digital currency          | 17.8175
Analog integrated circuit design Bob Pease | Bob Pease                 | 38.0711
Ammeter and voltmeter battery indication | Battery indicator         | 38.3279
Capacity fading in lithium-ion batteries | Capacity loss             | 30.7496
Secure military forum Iraq combat        | CAVNET                    | 29.1904


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 



In [73]:
for q in queries_taller:
    res_tfidf = recuperar_tfidf(q)
    res_bm25 = recuperar_bm25(q)
    
    docs_tfidf = {r.split('\n')[0][:30]: pos+1 for pos, r in enumerate(res_tfidf['raw'].values)}
    docs_bm25 = {r.split('\n')[0][:30]: pos+1 for pos, r in enumerate(res_bm25['raw'].values)}
    
    todos = list(dict.fromkeys(list(docs_tfidf.keys()) + list(docs_bm25.keys())))
    
    print(f"\nQuery: {q}")
    print(f"  {'Documento':<32} | {'Pos TF-IDF':<12} | {'Pos BM25'}")
    print("  " + "-" * 60)
    for doc in todos:
        pos_tf = str(docs_tfidf[doc]) if doc in docs_tfidf else "-"
        pos_bm = str(docs_bm25[doc]) if doc in docs_bm25 else "-"
        print(f"  {doc:<32} | {pos_tf:<12} | {pos_bm}")


Query: Renewable methanol production in Iceland
  Documento                        | Pos TF-IDF   | Pos BM25
  ------------------------------------------------------------
  Carbon Recycling International   | 1            | 1
  Reformed methanol fuel cell      | 2            | 8
  Ãslaug MagnÃºsdÃ³ttir           | 3            | -
  Renew                            | 4            | -
  Carbon-neutral fuel              | 5            | 2
  Ministry of Power and Renewabl   | 6            | -
  Journal of Renewable and Susta   | 7            | -
  George Philippidis               | 8            | -
  Renewable Fuels Regulators Clu   | 9            | 4
  Ministry of New and Renewable    | 10           | -
  Viaspace                         | -            | 3
  Low-carbon economy               | -            | 5
  Electrochemical reduction of c   | -            | 6
  Zytel                            | -            | 7
  Renewable fuels                  | -            | 9
  Oil depletion  

De la comparación se puede ver que BM25 tiende a colocar el documento más relevante en una posición más alta que TF-IDF. Por ejemplo, en la query "Chemical Agent Resistant Coating military", BM25 lo recuperó en posición 1 mientras que TF-IDF lo puso en posición 9. Esto se debe a que BM25 satura la frecuencia de los términos y penaliza documentos largos, lo que le ayuda a no darle peso excesivo a documentos que simplemente repiten mucho una palabra.

En cambio, ambos modelos coincidieron en el primer lugar en queries muy específicas como "Claas Cougar", "Bob Pease" o "Battery indicator", donde el documento tiene un nombre casi idéntico a la query.

En general, BM25 mostró mejores resultados para queries donde el documento relevante no necesariamente repite los términos muchas veces pero sí los contiene de forma precisa.